# Convert NetCDF to Zarr with Dynamic BBox Subsetting

This notebook reads a NetCDF file, applies a robust spatial `bbox` subset, and writes a chunked Zarr store.

It is designed to work with datasets covering different spatial scopes (for example Africa-wide and Morocco-only files).

In [2]:
from pathlib import Path

import importlib.util
import numpy as np
import xarray as xr

if importlib.util.find_spec("zarr") is None:
    raise ImportError(
        "Package 'zarr' is missing. Install it in atlas-env with:\n"
        "  conda install -n atlas-env -c conda-forge zarr numcodecs\n"
        "or:\n"
        "  pip install zarr numcodecs"
    )

# Avoid HTML rendering issues in Jupyter when dask templates are missing
xr.set_options(display_style="text")


In [3]:
# Auto-detect project root
project_root = Path.cwd()
if not (project_root / "data").exists():
    # If run from book/notebooks, go back to repo root
    project_root = project_root.parents[1]

data_dir = project_root / "data"

# Choose a specific folder inside data/ (set to None to search all data/)
preferred_data_subdir = "africa_era5"  # e.g. "africa_era5_precip", "africa_era5_evap"
search_dir = data_dir / preferred_data_subdir if preferred_data_subdir else data_dir

if not search_dir.exists():
    raise FileNotFoundError(f"Folder not found: {search_dir}")

netcdf_candidates = sorted(search_dir.rglob("*.nc"))
if not netcdf_candidates:
    raise FileNotFoundError(f"No NetCDF (*.nc) files found under: {search_dir}")

print(f"Search folder: {search_dir}")
print("Available NetCDF files:")
for i, pth in enumerate(netcdf_candidates):
    print(f"  [{i}] {pth.relative_to(project_root)}")

# Pick file by index in the list above
selected_file_index = 0
input_netcdf = netcdf_candidates[selected_file_index]
output_zarr = input_netcdf.with_suffix(".zarr")

print(f"Using input file: {input_netcdf}")
print(f"Output Zarr store: {output_zarr}")

# Bounding box in (lon_min, lon_max, lat_min, lat_max)
# Example Morocco-ish bbox:
bbox = (-13.5, -0.5, 27.0, 36.5)

# Chunking profile oriented for spatial queries
# Tune based on your variable sizes and access pattern
target_chunks = {"time": 30, "lat": 200, "lon": 200}


Search folder: /home/abdessamadelh/Documents/stage/projects/climate-notbooks/data/africa_era5
Available NetCDF files:
  [0] data/africa_era5/era5_africa_masked.nc
  [1] data/africa_era5/t_ERA5_Africa.nc
Using input file: /home/abdessamadelh/Documents/stage/projects/climate-notbooks/data/africa_era5/era5_africa_masked.nc
Output Zarr store: /home/abdessamadelh/Documents/stage/projects/climate-notbooks/data/africa_era5/era5_africa_masked.zarr


In [4]:
def detect_coord_name(ds, candidates):
    lower_map = {name.lower(): name for name in list(ds.coords) + list(ds.dims)}
    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]
    return None


def normalize_lon_value(lon, convention):
    if convention == "0_360":
        return lon % 360
    return ((lon + 180) % 360) - 180


def infer_lon_convention(lon_values):
    lon_min = float(np.nanmin(lon_values))
    lon_max = float(np.nanmax(lon_values))
    if lon_min >= 0 and lon_max > 180:
        return "0_360"
    return "-180_180"


def ordered_slice(coord_values, vmin, vmax):
    ascending = bool(coord_values[0] <= coord_values[-1])
    return slice(vmin, vmax) if ascending else slice(vmax, vmin)


def subset_bbox(ds, bbox):
    lon_min, lon_max, lat_min, lat_max = bbox

    lat_name = detect_coord_name(ds, ["lat", "latitude", "y"])
    lon_name = detect_coord_name(ds, ["lon", "longitude", "x"])

    if lat_name is None or lon_name is None:
        raise ValueError("Could not detect latitude/longitude coordinates in dataset.")

    lat_values = np.asarray(ds[lat_name].values)
    lon_values = np.asarray(ds[lon_name].values)

    lon_convention = infer_lon_convention(lon_values)
    lon_min_n = normalize_lon_value(lon_min, lon_convention)
    lon_max_n = normalize_lon_value(lon_max, lon_convention)

    lat_data_min, lat_data_max = sorted([float(np.nanmin(lat_values)), float(np.nanmax(lat_values))])
    lon_data_min, lon_data_max = sorted([float(np.nanmin(lon_values)), float(np.nanmax(lon_values))])

    lat_min_i = max(min(lat_min, lat_max), lat_data_min)
    lat_max_i = min(max(lat_min, lat_max), lat_data_max)

    if lat_min_i > lat_max_i:
        raise ValueError("Requested latitude bbox does not intersect dataset extent.")

    lat_sel = ordered_slice(lat_values, lat_min_i, lat_max_i)
    ds_lat = ds.sel({lat_name: lat_sel})

    # Handle antimeridian crossing after normalization
    if lon_min_n <= lon_max_n:
        lon_min_i = max(lon_min_n, lon_data_min)
        lon_max_i = min(lon_max_n, lon_data_max)
        if lon_min_i > lon_max_i:
            raise ValueError("Requested longitude bbox does not intersect dataset extent.")
        lon_sel = ordered_slice(lon_values, lon_min_i, lon_max_i)
        out = ds_lat.sel({lon_name: lon_sel})
    else:
        part1_min = max(lon_min_n, lon_data_min)
        part1_max = lon_data_max
        part2_min = lon_data_min
        part2_max = min(lon_max_n, lon_data_max)

        chunks = []
        if part1_min <= part1_max:
            chunks.append(ds_lat.sel({lon_name: ordered_slice(lon_values, part1_min, part1_max)}))
        if part2_min <= part2_max:
            chunks.append(ds_lat.sel({lon_name: ordered_slice(lon_values, part2_min, part2_max)}))

        if not chunks:
            raise ValueError("Requested longitude bbox does not intersect dataset extent.")
        out = xr.concat(chunks, dim=lon_name)

    return out, lat_name, lon_name

In [5]:
if not input_netcdf.exists():
    raise FileNotFoundError(f"Input file not found: {input_netcdf}")

# Open lazily for large files
ds = xr.open_dataset(input_netcdf, chunks="auto")

ds_subset, lat_name, lon_name = subset_bbox(ds, bbox)
print(f"Detected coordinates -> lat: {lat_name}, lon: {lon_name}")
ds_subset

ModuleNotFoundError: Failed to decode variable 'time': No module named 'dask.base'

In [ ]:
# Map generic chunk config to actual coordinate names in this file
chunks_for_ds = {}
for key, value in target_chunks.items():
    if key == "lat":
        chunks_for_ds[lat_name] = value
    elif key == "lon":
        chunks_for_ds[lon_name] = value
    elif key in ds_subset.dims:
        chunks_for_ds[key] = value

ds_chunked = ds_subset.chunk(chunks_for_ds) if chunks_for_ds else ds_subset
print(f"Chunks used: {chunks_for_ds}")

ds_chunked.to_zarr(output_zarr, mode="w", consolidated=True)
print(f"Zarr store written to: {output_zarr}")

Chunks used: {'time': 30, 'lat': 200, 'lon': 200}
Zarr store written to: /home/abdessamadelh/Documents/stage/projects/climate-notbooks/data/africa_era5/era5_africa_masked.zarr


In [ ]:
# Quick check: inspect chunking (with robust fallback if zarr engine is unavailable)
import math
import importlib.metadata as ilm

try:
    engines = xr.backends.list_engines()
except Exception:
    engines = {}

print("Available xarray engines:", list(engines))
for pkg in ["xarray", "zarr", "numcodecs", "dask"]:
    try:
        print(f"{pkg} version:", ilm.version(pkg))
    except Exception:
        print(f"{pkg} version: not installed")

# Rebuild effective chunk sizes used for this dataset
effective_chunks = {}
for key, value in target_chunks.items():
    if key == "lat":
        effective_chunks[lat_name] = value
    elif key == "lon":
        effective_chunks[lon_name] = value
    elif key in ds_subset.dims:
        effective_chunks[key] = value

if "zarr" in engines:
    ds_check = xr.open_zarr(output_zarr, consolidated=True)
    print("Reopened:", output_zarr)
else:
    print("zarr engine unavailable in this kernel -> fallback to in-memory dataset (ds_chunked).")
    ds_check = ds_chunked

print("Sizes:", dict(ds_check.sizes))
print("Effective chunk sizes:", effective_chunks)

# Chunk counts per dimension
chunk_counts_per_dim = {}
for dim, size in ds_check.sizes.items():
    chunk_size = effective_chunks.get(dim, size)
    chunk_counts_per_dim[dim] = math.ceil(size / chunk_size)
print("Chunk counts per dim:", chunk_counts_per_dim)

# Total chunks per variable (exact when chunks metadata is available)
for var_name, da in ds_check.data_vars.items():
    if da.chunks is None:
        print(f"{var_name}: unchunked")
        continue
    n_chunks = 1
    for axis_chunks in da.chunks:
        n_chunks *= len(axis_chunks)
    print(f"{var_name}: {n_chunks} chunks")

# Safe text preview (avoids Dask HTML template rendering issues)
print(ds_check)


Available xarray engines: ['netcdf4', 'scipy', 'cfgrib', 'earthkit', 'store']
xarray version: not installed
zarr version: not installed
numcodecs version: not installed
dask version: 2026.3.0
zarr engine unavailable in this kernel -> fallback to in-memory dataset (ds_chunked).
Sizes: {'time': 1020, 'lat': 39, 'lon': 53}
Effective chunk sizes: {'time': 30, 'lat': 200, 'lon': 200}
Chunk counts per dim: {'time': 34, 'lat': 1, 'lon': 1}
t: 34 chunks
crs: 1 chunks
<xarray.Dataset> Size: 8MB
Dimensions:   (time: 1020, lat: 39, lon: 53)
Coordinates:
  * lat       (lat) float64 312B 27.0 27.25 27.5 27.75 ... 35.75 36.0 36.25 36.5
  * lon       (lon) float64 424B -13.5 -13.25 -13.0 -12.75 ... -1.0 -0.75 -0.5
  * time      (time) datetime64[ns] 8kB 1940-01-01 1940-02-01 ... 2024-12-01
    height2m  float64 8B ...
Data variables:
    t         (time, lat, lon) float32 8MB dask.array<chunksize=(30, 39, 53), meta=np.ndarray>
    crs       (lat, lon) float64 17kB dask.array<chunksize=(39, 53), meta=